# 🧪 QTrader v4.7: Smart Paper Trading Engine (EV Diagnosis)

This notebook provides a safe, locally simulated trading environment using **real Coinbase market data**. It evaluates your strategy's Expected Value (EV), Win Rate, and Kelly Fraction before any real money is placed at risk.

### Rules for Going Live (`SIMULATE_MODE=False`):
1. **EV > 0**: Strategy must represent a mathematical edge.
2. **Win Rate > 52%**: Minimum threshold to reliably beat exchange fees.
3. **Slippage < 10 bps**: Avoid illiquid books.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent))

import polars as pl
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
from datetime import datetime, timedelta

from qtrader.core.config import Config
from qtrader.core.event import OrderEvent
from qtrader.input.data.market.coinbase_market import CoinbaseMarketDataClient
from qtrader.output.execution.paper_engine import PaperTradingEngine
from qtrader.output.analytics.ev_calculator import EVCalculator

# Set dark theme for Plotly
import plotly.io as pio
pio.templates.default = "plotly_dark"

## 1. Fetch Real Market Data

In [ ]:
SYMBOL = "BTC-USD"
capital = 10000.0

client = CoinbaseMarketDataClient()

# Fetch last 7 days of 1-hour candles
end_dt = datetime.utcnow()
start_dt = end_dt - timedelta(days=7)

candles = client.get_candles(SYMBOL, "ONE_HOUR", start=start_dt, end=end_dt)
print(f"Fetched {len(candles)} candles for {SYMBOL}")

book = client.get_product_book(SYMBOL, limit=1)
best_bid = book["bids"][0]["price"] if book["bids"] else 0.0
best_ask = book["asks"][0]["price"] if book["asks"] else 0.0
print(f"Current Top of Book: Bid {best_bid} | Ask {best_ask}")

## 2. Run Paper Trading Simulation (Mean Reversion Example)

In [ ]:
engine = PaperTradingEngine(starting_capital=capital)

# Very simple strategy: Buy if close < open, Sell if close > open
# This is purely to generate trades for the EV calculator.
for row in candles.iter_rows(named=True):
    market_state = {
        "bid": float(row["close"]) * 0.9999,
        "ask": float(row["close"]) * 1.0001,
        "top_depth": 5.0,
        "venue": "Coinbase_Sim"
    }
    
    qty = 0.01
    if row["close"] < row["open"]:
        # Buy signal
        order = OrderEvent(symbol=SYMBOL, order_type="MARKET", side="BUY", quantity=qty)
        engine.simulate_fill(order, market_state)
    elif row["close"] > row["open"]:
        # Sell signal
        order = OrderEvent(symbol=SYMBOL, order_type="MARKET", side="SELL", quantity=qty)
        engine.simulate_fill(order, market_state)
        
print(f"Simulation finished. Total closed trades: {len(engine.closed_trades)}")

## 3. Auto-Diagnosis & Expected Value

In [ ]:
calculator = EVCalculator(engine.closed_trades)
diagnosis = calculator.diagnose(SYMBOL)

print("=" * 40)
print(f"🧠 STRATEGY DIAGNOSIS: {diagnosis.status}")
print("=" * 40)
print(f"Total Trades:   {diagnosis.total_trades}")
print(f"Win Rate:       {diagnosis.win_rate:.2%}")
print(f"Expected Value: {diagnosis.ev_per_trade:.4f}")
print(f"Average Win:    {diagnosis.avg_win:.4f}")
print(f"Average Loss:   {diagnosis.avg_loss:.4f}")
print(f"Max Slippage:   {diagnosis.max_slippage_bps:.2f} bps")
print(f"Kelly Fraction: {diagnosis.kelly_fraction:.4f}")

if diagnosis.warnings:
    print("\n⚠️ WARNINGS:")
    for w in diagnosis.warnings:
        print(f" - {w}")

## 4. Visualizations

In [ ]:
if engine.closed_trades:
    df_trades = pd.DataFrame([vars(t) for t in engine.closed_trades])
    df_trades['cumulative_pnl'] = df_trades['pnl'].cumsum()
    
    fig = px.line(df_trades, y='cumulative_pnl', title='Paper Trading Cumulative PnL',
                  labels={'index': 'Trade #', 'cumulative_pnl': 'PnL (USD)'})
    fig.update_layout(title_font_size=20)
    fig.show()
    
    # Win/Loss scatter
    fig2 = px.scatter(df_trades, y='pnl', color=(df_trades['pnl'] > 0), 
                      title='Trade Profitability Spread',
                      labels={'color': 'Profitable', 'index': 'Trade #', 'pnl': 'PnL (USD)'})
    fig2.show()
else:
    print("No trades to visualize.")